# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features 
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Imports

In [1]:
#Imports
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

#import joypy
from scipy import stats

from plate_information import *
from plate_preprocessing import *


## Get table info for a single plate

In [ ]:
root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v3_active/"
filename = "20250501_output.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250501_rep07"

plate_dfs = {}
nuclei_dfs = {}

pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
#metadata extraction
map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
print(map_file)

cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
metadata_cols = [col for col in cell_df.columns if "Metadata" in col]
#display(cell_df)

#Remove null/infinite rows
cols_to_check = ['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field']
cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
cell_df = cell_df.dropna(subset=cols_to_check)        # Drop rows with NaN in these columns
#display(cell_df)

#Find cell/nuc area ratio
cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(cell_df, "Cell_AreaShape_Area")
#get rid of cells where area is lower than nuc area
cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1]
cell_df["Cell_Nuclei_Area_Ratio"].plot(kind="density",xlim=(-1,200))
display(
    cell_df[
        [
            "Cell_AreaShape_Area",
            "Cell_Mean_Nuclei_AreaShape_Area",
            "Cell_Children_Nuclei_Count",
            "Cell_Nuclei_Area_Ratio",
        ]
    ]
)

#make these metadatas string

cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
display(cell_df[metadata_cols])

cell_df.reset_index(drop=True)

#add the median data
#cell_df = load_organelle_medians(db_path, cell_df)

print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)
    
    platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
    platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
    platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
    #platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
    #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
    
    #display(cell_df[cell_df["Metadata_Well"] == "A02"])
    
    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]
    
min_x = 0
min_y = 0
max_x = cell_df["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = cell_df["Image_Height_DAPI"][0]
        
cell_df_excluded_borders = exclude_borders(cell_df, min_x, min_y, max_x, max_y, prefix="Cell_")

plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
#display(combined_cell_df)

# Exporting zone: 
Make sure you put "_active" in the CP output folder names!

In [ ]:
# Setting file paths
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"
parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output"

parent_object_table = "Per_Cell"
compartment_tables = [ 'Per_MergedNucleiPerCell','Per_MergedMitoPerCell','Per_MergedLysoPerCell'] #note that these must be in a 1:1 relationship with cell

# query designed to remove cells from the dataset without any mitochondria, nuclei or lysosomes; allows us to have a 1:1 relationship
parent_obj_query = f"SELECT * FROM Per_{parent_object_table} WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0 AND Cell_Children_Nuclei_Count > 0;"

# Initialize a list to store the combined DataFrames
plate_dfs = {}
nuclei_dfs = {}


In [ ]:
def load_and_combine_plates_from_db(parent_dir,curr_plates,compartment_name= "Cell"):
# Loop over the plates
    for root, dirs, files in os.walk(parent_dir):
        for filename in files: 
            if filename.endswith(".db") and "active" in root and "extraRow1" not in filename:
                db_path = os.path.join(root, filename) #make the path
                for plate in curr_plates:
                    if plate in db_path:
                        conn = sqlite3.connect(db_path)
                        cursor = conn.cursor()
                        #update_database_with_well_metadata(db_path)
                        try:
                            # Read the 'Per_Cell' table and get metadata from 'Per_Image' table
                            pre_cell_df = pd.read_sql_query(f"SELECT * FROM Per_{compartment_name}", conn)
                            image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
                            #metadata extraction - make sure its the exact same file format as the one above
                            map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
                            print(map_file)
                            
                            cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
                            cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
                            
                            #remove rows where there isn't a valid row/column/field metadata
                            cols_to_check = ['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field']
                            cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
                            cell_df = cell_df.dropna(subset=cols_to_check)        # Drop rows with NaN in these columns
                            
                            #make these metadatas string
                            cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
                            cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
                            cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
                            
                            #add the median data
                            cell_df = load_organelle_medians(db_path, cell_df)
                            cell_df.reset_index(drop=True)
                            
                            #Find cell/nuc area ratio
                            cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(cell_df, "Cell_AreaShape_Area")
                            #get rid of cells where area is lower than nuc area
                            cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1].reset_index(drop=True)
                            
                            print(os.path.exists(map_file))
                            
                            if os.path.exists(map_file):
                                platemap_df = pd.read_csv(map_file)
                                display(platemap_df)
                                platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
                                platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
                                platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
                                
                                #platemap_df.reset_index(drop=True)
                                cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
                                #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
                                cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
                                cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
                                
                                cell_df["Metadata_Plate"] = plate
                                cell_df["Replicate_Number"] = plate[-1]
                                cell_df["Replicate_String"] = f"R{plate[-1]}"
                                
                            plate_dfs[plate] = cell_df
                        except Exception as e:
                            print(f"Error reading {db_path}: {e}")
                        finally:
                            conn.close()

    # Combine all DataFrames
    combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
    return combined_cell_df


In [ ]:
combined_cell_df = load_and_combine_plates_from_db(parent_dir,curr_plates)
#combined_nuclei_df = pd.concat(nuclei_dfs.values(), ignore_index=True)
                
#Filter DataFrames to only include cells that were stained with LAMP1-488 and MitoRed

#combined_nuclei_df_mitolyso = combined_nuclei_df[combined_nuclei_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]



## Export everything to CSV

In [ ]:
#Export to a giant csv
# filter out the non-experimental test images
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]

display(combined_cell_df_mitolyso.head(10))
# print(cell_df.shape, " ", filter_df.shape)

#export the plate
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
combined_cell_df_mitolyso.to_csv(
    os.path.join(outpath, "total_combined_cell.csv"), index=False
)

#set up the image borders and make a csv with excluded border cells
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

combined_cell_df_mitolyso_borders_excluded = exclude_borders(
    combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_"
)
combined_cell_df_mitolyso_borders_excluded.to_csv(
    os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False
)


### Summary Stats

# Make Feature Lists here:

In [ ]:

#file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'


columns_list = define_cell_features(combined_cell_df_mitolyso_merged)

mito_features = make_feature_dict([col for col in columns_list if 'Mito' in col])
lyso_features = make_feature_dict([col for col in columns_list if 'Lysosome' in col or 'LAMP1' in col or 'Lyso' in col])
nuc_features = make_feature_dict([col for col in columns_list if 'Nuc' in col or 'DAPI' in col])
print(columns_list)


In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"- {feature}\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


sns.pairplot(cell_df, hue='Passage Group', vars=mito_features['radialdistribution'], diag_kind='kde', plot_kws={'alpha':0.5})
plt.show()

## Normalize features to control (Passage 6-8)

In [ ]:
norm_cell_df = combined_cell_df_mitolyso_merged.copy()
norm_cell_df_nuc = apply_feature_normalization(norm_cell_df, nuc_features, curr_plates).copy()
norm_cell_df_mito = apply_feature_normalization(norm_cell_df_nuc, mito_features, curr_plates).copy()
norm_cell_df_mitolyso = apply_feature_normalization(norm_cell_df_mito, lyso_features, curr_plates)

#watch out - merged df might be clipping off all of the features
